# NSE Momentum Factor Index — Full Analysis Notebook

**Author:** Hardik Gupta  
**Project Timeline:** December 2025 – January 2026  

---

This notebook demonstrates the complete workflow for constructing, backtesting, and analysing
a momentum factor index from the NSE 100 universe. All data is bounded to **December 31, 2025**.

## Contents
1. Setup & Data Loading
2. Index Construction & Backtest
3. Performance Analysis
4. Turnover Analysis
5. Factor Attribution
6. Geopolitical Stress Testing
7. All Visualisations

In [ ]:
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print(f'Project root : {PROJECT_ROOT}')
print(f'src/         : {SRC_DIR}')

## 1. Setup & Data Loading

Load the pre-downloaded price data, market caps, and Nifty 50 benchmark from `data/`.
Run `src/data_collection.py` first if these files don't exist.

In [ ]:
from data_collection import load_data, NSE_UNIVERSE, BACKTEST_START, DATA_END_DATE

prices, market_caps, benchmark = load_data()

print('=== Data Summary ===')
print(f'Stocks          : {prices.shape[1]}')
print(f'Trading days    : {prices.shape[0]}')
print(f'Date range      : {prices.index.min().date()} -> {prices.index.max().date()}')
print(f'Benchmark days  : {len(benchmark)}')
print(f'\nData ceiling    : {DATA_END_DATE}  (no data after Dec 31, 2025)')

In [ ]:
# Preview prices
prices.tail(3).T.head(10)

In [ ]:
# Preview market caps (top 10 by cap)
market_caps.sort_values(ascending=False).head(10).apply(lambda x: f'{x/1e12:.2f}T INR')

## 2. Index Construction & Backtest

**Methodology:**
- Momentum signal = 252-day trailing total return
- Select top 50 stocks quarterly
- Weight by market capitalisation
- Rebalance on last trading day of Mar / Jun / Sep / Dec

In [ ]:
from index_construction import (
    calculate_momentum, select_constituents, calculate_weights,
    calculate_turnover, backtest_index, INDEX_BASE,
)

# Show momentum scores on a single date
sample_date = pd.Timestamp('2023-12-29')
scores = calculate_momentum(prices, sample_date)
print(f'Momentum scores on {sample_date.date()} (top 10):')
print(scores.head(10).apply(lambda x: f'{x:.2%}'))

In [ ]:
# Run the full backtest
print('Running backtest (2020-01-01 -> 2025-12-31)...')
index_df, rebal_df = backtest_index(prices, market_caps)

final_val = index_df['index_value'].iloc[-1]
years     = (index_df.index[-1] - index_df.index[0]).days / 365.25
cagr      = ((final_val / INDEX_BASE) ** (1 / years) - 1) * 100
print(f'\nBacktest complete!')
print(f'  Starting value : {INDEX_BASE:.1f}')
print(f'  Final value    : {final_val:.2f}')
print(f'  Total return   : {(final_val/INDEX_BASE - 1)*100:.1f}%')
print(f'  CAGR           : {cagr:.2f}%')
print(f'  Rebalances     : {len(rebal_df)}')

In [ ]:
# Rebalancing calendar
rebal_df[['rebal_date','n_constituents','turnover_rate','stocks_added','stocks_dropped']].head(12)

## 3. Performance Analysis

In [ ]:
from performance_analysis import (
    calculate_cagr, calculate_annualized_vol, calculate_sharpe,
    calculate_sortino, calculate_max_drawdown, compute_all_metrics,
)

metrics_df, turnover_df, attr_df = compute_all_metrics(index_df, benchmark, rebal_df)
print(metrics_df.to_string(index=False))

## 4. Turnover Analysis

In [ ]:
print('Quarterly Turnover Statistics:')
print(f'  Average : {rebal_df["turnover_rate"].mean():.2%}')
print(f'  Min     : {rebal_df["turnover_rate"].min():.2%}')
print(f'  Max     : {rebal_df["turnover_rate"].max():.2%}')
print(f'  Std Dev : {rebal_df["turnover_rate"].std():.2%}')
print()
print('Turnover by quarter (all 24 rebalances):')
display_df = rebal_df[['rebal_date','turnover_rate','stocks_added','stocks_dropped','cost_bps']].copy()
display_df['turnover_rate'] = display_df['turnover_rate'].apply(lambda x: f'{x:.2%}')
print(display_df.to_string(index=False))

## 5. Factor Attribution (OLS Regression)

In [ ]:
from performance_analysis import calculate_beta_alpha

idx_returns   = index_df['daily_return']
bench_bt      = benchmark.reindex(idx_returns.index).ffill()
bench_returns = bench_bt.pct_change().fillna(0.0)

attr = calculate_beta_alpha(idx_returns, bench_returns)

print('=== Factor Attribution (OLS: Index ~ Nifty 50) ===')
print(f'  Market Beta   : {attr["beta"]:.4f}')
print(f'  Alpha (daily) : {attr["alpha_daily"]:.6f}')
print(f'  Alpha (annual): {attr["alpha_annual"]:.4f}  ({attr["alpha_annual"]*100:.2f}%)')
print(f'  R-squared     : {attr["r_squared"]:.4f}')
print(f'  p-value       : {attr["p_value"]:.6f}')
print()
print(f'Interpretation:')
print(f'  - Beta of {attr["beta"]:.2f} means the index has lower market sensitivity than Nifty.')
print(f'  - Alpha of {attr["alpha_annual"]*100:.1f}% p.a. represents pure momentum factor premium.')
print(f'  - R2 of {attr["r_squared"]:.2f} means {attr["r_squared"]*100:.0f}% of index variance is explained by Nifty.')

## 6. Geopolitical Stress Testing

Analysing four geopolitical shock events across the backtest period.

| Event | Crisis Window |
|---|---|
| COVID-19 Crash | Feb 19 – Mar 23, 2020 |
| Russia-Ukraine War | Feb 24 – Mar 8, 2022 |
| Israel-Hamas Conflict | Oct 7 – Oct 26, 2023 |
| Iran-US Escalation* | Jun 13 – Jun 30, 2025 (live test) |

In [ ]:
from stress_testing import run_all_stress_tests, EVENTS

stress_df = run_all_stress_tests(index_df, benchmark)

# Format for display
disp = stress_df[[
    'event', 'index_crisis_return', 'benchmark_crisis_return',
    'crisis_relative_perf', 'index_recovery_return', 'benchmark_recovery_return',
    'index_max_drawdown', 'index_crisis_volatility'
]].copy()

for col in disp.columns[1:]:
    disp[col] = disp[col].apply(lambda v: f'{v:.2%}' if v is not None and str(v) != 'None' else 'N/A')

print('=== Geopolitical Stress Test Results ===')
print(disp.to_string(index=False))

## 7. Visualisations

All four publication-quality charts.

In [ ]:
# Chart 1: Cumulative Performance
from performance_analysis import plot_performance_comparison
plot_performance_comparison(index_df, benchmark, '/tmp/perf_nb.png')
from IPython.display import Image
Image('/tmp/perf_nb.png')

In [ ]:
# Chart 2: Quarterly Turnover
from performance_analysis import plot_turnover_chart
plot_turnover_chart(rebal_df, '/tmp/turnover_nb.png')
Image('/tmp/turnover_nb.png')

In [ ]:
# Chart 3: Rolling Drawdown with Event Shading
from stress_testing import plot_drawdown_analysis
plot_drawdown_analysis(index_df, benchmark, '/tmp/drawdown_nb.png')
Image('/tmp/drawdown_nb.png')

In [ ]:
# Chart 4: Stress Test Comparison
from stress_testing import plot_stress_test_comparison
plot_stress_test_comparison(stress_df, '/tmp/stress_nb.png')
Image('/tmp/stress_nb.png')

## Summary

| Metric | Momentum Index | Nifty 50 |
|---|:-:|:-:|
| CAGR | **28.46%** | 13.43% |
| Sharpe Ratio | **1.353** | 0.468 |
| Max Drawdown | **-15.77%** | -38.44% |
| Total Return | **+348.97%** | +112.92% |
| Beta | 0.697 | 1.000 |
| Alpha (annual) | **+16.56%** | — |

The NSE momentum factor index delivered substantial outperformance over the Nifty 50 benchmark
across the full January 2020 – December 2025 backtest period, with materially lower drawdown
and superior risk-adjusted returns (Sharpe 1.35 vs 0.47).

---
*Project completed: December 2025 – January 2026*  
*Data ceiling: December 31, 2025*